In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/bhaskarreddy072/mail-datacsv/mail_data.csv


In [2]:
# Importing important libraries for data handling, visualization, and machine learning

import numpy as np                  # Used for numerical operations and working with arrays
import pandas as pd                 # Used for data manipulation and analysis

import seaborn as sns              # Used for creating attractive statistical visualizations
import matplotlib.pyplot as plt    # Used for plotting graphs and charts

# Function to split dataset into training data and testing data
from sklearn.model_selection import train_test_split

# Logistic Regression model used for classification problems
from sklearn.linear_model import LogisticRegression

# Converts text data into numerical feature vectors using TF-IDF technique
from sklearn.feature_extraction.text import TfidfVectorizer
# TfidfVectorizer automatically:

# 1. Splits text into words
# 2. Removes punctuation
# 3. Builds vocabulary
# 4. Counts words
# 5. Calculates TF
# 6. Calculates IDF
# 7. Multiplies TF × IDF
# 8 .Creates numerical matrix

# All in one step.

# Used to evaluate the accuracy of the machine learning model
from sklearn.metrics import accuracy_score

# Used Labelencoding to dataset as there is ham/spam

from sklearn.preprocessing import LabelEncoder

In [3]:
# Flow of the TF-IDF calculation

# Raw Text
#    ↓
# Cleaning Text
#    ↓
# Tokenization
#    ↓
# TF-IDF Calculation
#    ↓
# Numerical Matrix
#    ↓
# Machine Learning Model



# Step 1 — Understanding "Term Frequency (TF)"
#     TF measures: "How many times a word appears in a document."
#         Formula: TF(t)= Total number of terms in the document / Number of times term t appears in a document

# Step 2 — Understanding "Inverse Document Frequency (IDF)"
#     IDF measures: "How rare or unique a word is across all documents."
#     If a word appears in almost every document:
#         It is less important
#     If a word appears in very few documents:
#         It is more important
#     formula : IDF(t)=log(Number of documents containing term t / Total number of documents)
# Step 3 - Final TF-IDF Formula
#         TF-IDF combines both concepts:
#         TF-IDF(t)=TF(t)×IDF(t)
#         example : Importance =
#                 (How often word appears in this document)
#                     ×
#                 (How rare the word is overall)

# TF-IDF IMPORTANT PARAMETERS

# stop_words='english'
# Removes common English words like:
# "the", "is", "and", "a"
# Helps reduce unnecessary noise in text data


# max_features=5000
# Keeps only the top 5000 most important words
# Useful for reducing dimensionality and improving performance


# ngram_range=(1,2)
# (1,1) -> Unigrams (single words)
# (1,2) -> Unigrams + Bigrams
# Example:
# "machine"
# "machine learning"
# Helps capture word relationships and context


# min_df=2
# Ignores words that appear in less than 2 documents
# Removes very rare/unimportant words


# max_df=0.8
# Removes words appearing in more than 80% of documents
# Helps eliminate overly common words


# lowercase=True
# Converts all text into lowercase
# "Python" and "python" become the same word


# token_pattern
# Controls how words/tokens are identified from text
# Default ignores punctuation and special symbols


# sublinear_tf=True
# Applies logarithmic scaling to term frequency
# Prevents very frequent words from dominating


# use_idf=True
# Enables Inverse Document Frequency weighting
# Gives higher importance to rare words


# smooth_idf=True
# Prevents division-by-zero errors in IDF calculation
# Adds stability to the formula


# norm='l2'
# Normalizes vectors for better ML performance
# Commonly used in cosine similarity and classification


# binary=True
# Uses only presence/absence of words
# Instead of actual word frequency counts

In [4]:
# Loading the dataset from the CSV file using pandas

dataset = pd.read_csv("/kaggle/input/datasets/bhaskarreddy072/mail-datacsv/mail_data.csv")

# Displays the first 5 rows of the dataset
# Helps us quickly understand the structure, columns, and sample data

dataset.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
# Replacing all missing (null/NaN) values in the dataset with empty strings ('')
# pd.notnull(dataset) checks for non-null values
# .where() keeps existing values if they are not null
# If a value is null, it gets replaced with ''

new_dataset = dataset.where((pd.notnull(dataset)), '')
new_dataset = pd.DataFrame(new_dataset)
new_dataset.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
# Displays the number of rows and columns in the dataset
# Output format:
# (number_of_rows, number_of_columns)

new_dataset.shape

(5572, 2)

In [7]:
# Converting categorical labels into numerical values manually

# Replacing 'spam' with 1
# 1 represents spam/fake/unwanted email

new_dataset.loc[new_dataset['Category'] == 'spam', 'Category'] = 1

# Replacing 'ham' with 0
# 0 represents normal/non-spam email

new_dataset.loc[new_dataset['Category'] == 'ham', 'Category'] = 0

# Displays the first 5 rows after label conversion
# Helps verify whether encoding was done correctly

new_dataset.head()

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
# Splitting the dataset into features (input data) and target (output label)

# x contains the input features
# Dropping the 'Category' column because it is the target/output column
# axis=1 indicates column-wise dropping

x = new_dataset['Message']

# y contains the target values (labels)
# This is what the model will try to predict
# In spam detection:
# 0 -> ham
# 1 -> spam

y = new_dataset['Category']

In [9]:
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=123, test_size=0.2)

In [10]:
# Creating an object of TfidfVectorizer
# Used to convert text messages into numerical feature vectors

feature_extraction = TfidfVectorizer(

    stop_words='english',  
    # Removes common English words like:
    # "the", "is", "and", "a"
    # Helps reduce unnecessary noise in the text

    min_df=1,
    # Keeps words that appear in at least 1 document
    # Rare words below this threshold are ignored

    lowercase=True,
    # Converts all text into lowercase
    # Example:
    # "FREE" and "free" are treated as the same word

    use_idf=True,
    # Enables Inverse Document Frequency weighting
    # Gives higher importance to rare and meaningful words

    ngram_range=(1,2)
    # (1,1) -> Unigrams (single words)
    # (1,2) -> Unigrams + Bigrams
    # Example:
    # "free"
    # "free money"
    # Helps capture context and word relationships
)

In [11]:
# Converting text data into TF-IDF numerical feature vectors

# fit_transform() is used on training data because:
# 1. fit() learns:
#    - vocabulary
#    - important words
#    - IDF (Inverse Document Frequency) values
#
# 2. transform() converts text into numerical vectors
#
# The model should learn ONLY from training data

x_train_features = feature_extraction.fit_transform(x_train)

# transform() is used on testing data because:
# We should NOT learn anything from test data
#
# Using fit_transform() on test data can cause DATA LEAKAGE
# which means the model indirectly gets information from test data
#
# So we only transform test data using the vocabulary
# already learned from training data

x_test_features = feature_extraction.transform(x_test)

# Converting target labels into integer datatype
# Required for machine learning model training

y_train = y_train.astype('int')
y_test = y_test.astype('int')

In [12]:
model = LogisticRegression()
model.fit(x_train_features, y_train)


LogisticRegression()

In [13]:
# Prediction on training data

train_prediction = model.predict(x_train_features)

# Calculating training accuracy

train_accuracy = accuracy_score(y_train, train_prediction)

print("Training Accuracy:", train_accuracy * 100)

Training Accuracy: 95.17612743998205


In [14]:
# Prediction on the testing data

test_prediction = model.predict(x_test_features)

# Calculatiog testing accuracy 

test_accuracy = accuracy_score(y_test, test_prediction)

print("Testing Accuracy: ", test_accuracy * 100)

Testing Accuracy:  96.23318385650225


In [15]:
print(type(x_train))
print(x_train.shape)

print(type(y_train))
print(y_train.shape)

<class 'pandas.core.series.Series'>
(4457,)
<class 'pandas.core.series.Series'>
(4457,)


In [16]:
print(new_dataset['Category'].unique())
print(y_train.value_counts())

[0 1]
Category
0    3863
1     594
Name: count, dtype: int64


In [18]:
input_mail = ['URGENT! Your account has been suspended. Verify your bank details immediately to avoid permanent closure. Click here to claim your reward and win $5000 cash now!']

input_data_features = feature_extraction.transform(input_mail)
input_prediction = model.predict(input_data_features)
if input_prediction[0] == 1:
    print("Spam mail!! Ignore the mail")
else :
    print("Ham mail!! Open to read")

Spam mail!! Ignore the mail


In [19]:
input_mail = ['Hi jane how are you? Please call me when you are free.']

input_data_features = feature_extraction.transform(input_mail)
input_prediction = model.predict(input_data_features)
if input_prediction[0] == 1:
    print("Spam mail!! Ignore the mail")
else :
    print("Ham mail!! Open to read")

Ham mail!! Open to read


In [21]:
input_mail = [
    'Free entry in 2 a weekly competition to win FA Cup final tickets. Text WIN to 87121 now!'
]

input_data_features = feature_extraction.transform(input_mail)
input_prediction = model.predict(input_data_features)
if input_prediction[0] == 1:
    print("Spam mail!! Ignore the mail")
else :
    print("Ham mail!! Open to read")

Spam mail!! Ignore the mail
